In [0]:
# Importing packages and initiating variables

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, lit
import pandas as pd

port = 1433
server = "xlc-azu-eus2-eds-prd-dw-server01.database.windows.net"
database = "dw_xle_sz"
synapse_schema_suffix, adls_schema_suffix = "", "_polybase"
schemas = ["cad"]

In [0]:
# Function to run SELECT query on Synapse
def run_synapse_query(query):
  jdbcUrl = f"jdbc:sqlserver://{server}:{port};database={database};encrypt=true;trustServerCertificate=true;hostNameInCertificate=*.database.windows.net;loginTimeout=30"

  df = spark.read \
    .format("jdbc") \
    .option("url", jdbcUrl) \
    .option("query", query) \
    .option("user", "") \
    .option("password", "") \
    .load()

  return df


In [0]:
# Function to compare DataFrames (Table Structure)

def compare_table_structure(df1, df2):
  compared_df = (df1.withColumnRenamed("data_type", "adls_data_type")
                .join(df2.withColumnRenamed("data_type", "synapse_data_type"),
                      on = ["table_name", "col_name"], how = "inner")
                ).filter(col("synapse_data_type") != col("adls_data_type"))

  differences = {}
  for row in compared_df.collect():
    if row["adls_data_type"] != row["synapse_data_type"]:
      differences[row["col_name"]] = f"{row['synapse_data_type']} -> {row['adls_data_type']}"

  return str(differences)

In [0]:
# Compare rows, columns, data types and data for each table
products, table_names, synapse_size, adls_size, col_dtypes, syn_adls_rows, adls_syn_rows = [], [], [], [], [], [], []

for schema in schemas:
  synapse_schema, adls_schema = f"{schema}{synapse_schema_suffix}", f"{schema}{adls_schema_suffix}"
  
  table_query = f"""SELECT TABLE_NAME AS table_name FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = '{adls_schema}' """
  tables = run_synapse_query(table_query)
  
  for table in tables.collect():
    table_name = table["table_name"]

    try:
      synapse_row_count = run_synapse_query(f"SELECT COUNT(1) AS count FROM [{synapse_schema}].[{table_name}]").collect()[0]["count"]
      synapse_column_query = f"""SELECT
        TABLE_NAME AS table_name,
        COLUMN_NAME AS col_name, 
        CASE DATA_TYPE
          WHEN 'int' THEN DATA_TYPE
          WHEN 'char' THEN CONCAT(DATA_TYPE, '(', TRIM(STR(CHARACTER_MAXIMUM_LENGTH)), ')' )
          WHEN 'varchar' THEN CONCAT(DATA_TYPE, '(', TRIM(STR(CHARACTER_MAXIMUM_LENGTH)) ,')' )
          WHEN 'decimal' THEN CONCAT(DATA_TYPE, '(', TRIM(STR(NUMERIC_PRECISION)), ',', TRIM(STR(NUMERIC_SCALE)), ')' )
          ELSE DATA_TYPE END AS data_type
        FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = '{table_name}' AND TABLE_SCHEMA = '{synapse_schema}' """
      synapse_columns = run_synapse_query(f"{synapse_column_query}")
      synapse_col_count = synapse_columns.count()
      synapse_rc = (f"({synapse_row_count}, {synapse_col_count})")
      synapse_adls_data_match_query = f"""SELECT * FROM [{synapse_schema}].[{table_name}] EXCEPT SELECT * FROM [{adls_schema}].[{table_name}]"""
      synapse_adls_data = run_synapse_query(synapse_adls_data_match_query).count()
      compared_columns = "-"
    
    except:
      synapse_row_count, synapse_col_count, compared_columns, adls_synapse_data, synapse_adls_data = "-", "-", "{}", "-", "-"

    adls_row_count = run_synapse_query(f"SELECT COUNT(1) AS count FROM [{adls_schema}].[{table_name}]").collect()[0]["count"]
    adls_column_query = f"""SELECT
      TABLE_NAME AS table_name,
      COLUMN_NAME AS col_name, 
      CASE DATA_TYPE
        WHEN 'int' THEN DATA_TYPE
        WHEN 'char' THEN CONCAT(DATA_TYPE, '(', TRIM(STR(CHARACTER_MAXIMUM_LENGTH)), ')' )
        WHEN 'varchar' THEN CONCAT(DATA_TYPE, '(', TRIM(STR(CHARACTER_MAXIMUM_LENGTH)) ,')' )
        WHEN 'decimal' THEN CONCAT(DATA_TYPE, '(', TRIM(STR(NUMERIC_PRECISION)), ',', TRIM(STR(NUMERIC_SCALE)), ')' )
        ELSE DATA_TYPE END AS data_type
      FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = '{table_name}' AND TABLE_SCHEMA = '{adls_schema}' """
    adls_columns = run_synapse_query(f"{adls_column_query}")
    adls_col_count = adls_columns.count()
    adls_rc = (f"({adls_row_count}, {adls_col_count})")

    if compared_columns == "-":
      adls_synapse_data_match_query = f"""SELECT * FROM [{adls_schema}].[{table_name}] EXCEPT SELECT * FROM [{synapse_schema}].[{table_name}]"""
      adls_synapse_data = run_synapse_query(adls_synapse_data_match_query).count()
      compared_columns = compare_table_structure(adls_columns, synapse_columns)
    
    products.append(f"{schema}")
    table_names.append(f"{table_name}")
    synapse_size.append(synapse_rc)
    adls_size.append(adls_rc)
    col_dtypes.append(compared_columns)
    syn_adls_rows.append(synapse_adls_data)
    adls_syn_rows.append(adls_synapse_data)

data = {
  'product': products,
  'table_name': table_names,
  'synapse_size': synapse_size,
  'adls_size': adls_size,
  'synapse_adls_dtypes_mismatch': col_dtypes,
  'synapse-adls_mismatch_rows': syn_adls_rows,
  'adls-synapse_mismatch_rows': adls_syn_rows
  }

final_df = pd.DataFrame(data)
display(final_df)

--------------------------------------------------------------------------- 
 Py4JJavaError Traceback (most recent call last)
 <command-1381023441571321> in <module> 
 6 
 7 table_query = f"""SELECT TABLE_NAME AS table_name FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = '{adls_schema}' """ 
 ----> 8 tables = run_synapse_query ( table_query ) 
 9 
 10 for table in tables . collect ( ) : 

 <command-1381023441571319> in run_synapse_query (query) 
 3 jdbcUrl = f"jdbc:sqlserver://{server}:{port};database={database};encrypt=true;trustServerCertificate=true;hostNameInCertificate=*.database.windows.net;loginTimeout=30" 
 4 
 ----> 5 df = spark . read \ 
 6 . format ( "jdbc" ) \ 
 7 . option ( "url" , jdbcUrl ) \ 

 /databricks/spark/python/pyspark/sql/readwriter.py in load (self, path, format, schema, **options) 
 162 return self . _df ( self . _jreader . load ( self . _spark . _sc . _jvm . PythonUtils . toSeq ( path ) ) ) 
 163 else : 
 --> 164 return self . _df ( self . _jreader . load ( ) ) 
 165 
 166 def json(self, path, schema=None, primitivesAsString=None, prefersDecimal=None,

 /databricks/spark/python/lib/py4j-0.10.9.1-src.zip/py4j/java_gateway.py in __call__ (self, *args) 
 1302 
 1303 answer = self . gateway_client . send_command ( command ) 
 -> 1304 return_value = get_return_value(
 1305 answer, self.gateway_client, self.target_id, self.name)
 1306 

 /databricks/spark/python/pyspark/sql/utils.py in deco (*a, **kw) 
 115 def deco ( * a , ** kw ) : 
 116 try : 
 --> 117 return f ( * a , ** kw ) 
 118 except py4j . protocol . Py4JJavaError as e : 
 119 converted = convert_exception ( e . java_exception ) 

 /databricks/spark/python/lib/py4j-0.10.9.1-src.zip/py4j/protocol.py in get_return_value (answer, gateway_client, target_id, name) 
 324 value = OUTPUT_CONVERTER [ type ] ( answer [ 2 : ] , gateway_client ) 
 325 if answer [ 1 ] == REFERENCE_TYPE : 
 --> 326 raise Py4JJavaError(
 327 "An error occurred while calling {0}{1}{2}.\n" . 
 328 format(target_id, ".", name), value)

 Py4JJavaError : An error occurred while calling o588.load.
: com.microsoft.sqlserver.jdbc.SQLServerException: Login failed for user ''. ClientConnectionId:d9379c94-7971-42d0-90b9-9610f08025a9
	at com.microsoft.sqlserver.jdbc.SQLServerException.makeFromDatabaseError(SQLServerException.java:262)
	at com.microsoft.sqlserver.jdbc.TDSTokenHandler.onEOF(tdsparser.java:283)
	at com.microsoft.sqlserver.jdbc.TDSParser.parse(tdsparser.java:129)
	at com.microsoft.sqlserver.jdbc.TDSParser.parse(tdsparser.java:37)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.sendLogon(SQLServerConnection.java:5333)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.logon(SQLServerConnection.java:4066)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.access$000(SQLServerConnection.java:85)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection$LogonCommand.doExecute(SQLServerConnection.java:4004)
	at com.microsoft.sqlserver.jdbc.TDSCommand.execute(IOBuffer.java:7418)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.executeCommand(SQLServerConnection.java:3272)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.connectHelper(SQLServerConnection.java:2768)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.login(SQLServerConnection.java:2418)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.connectInternal(SQLServerConnection.java:2265)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.connect(SQLServerConnection.java:1291)
	at com.microsoft.sqlserver.jdbc.SQLServerDriver.connect(SQLServerDriver.java:881)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.$anonfun$createConnectionFactory$1(JdbcUtils.scala:69)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD$.getQueryOutputSc

In [0]:

synapse_schema = 'gismo_btl'
table_name = 't_ax_supp_policy_details'
adls_schema = 'dw_btl_gismo'
table_name_adls = 't_ax_supp_policy_details'
# table_query = f"""SELECT TABLE_NAME AS table_name FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = '{synapse_schema}' """
table_query = f"""select * from {synapse_schema}.{table_name}"""
df = run_synapse_query(table_query)
df.createOrReplaceTempView(f"""{table_name}_vw_rs""")





#### compare the columns
synapse_tbl_column = df.columns

df = spark.sql("describe table dw_btl_rdapp.claim_metrics")
adls_column_list = [row.col_name for row in df.select('col_name').collect()]
diff_columns = set(synapse_tbl_column) - set(adls_column_list)
if diff_columns:
  print('column mismatch, the difference', diff_columns)

syn_count = run_synapse_query(f'select * from {synapse_schema}.{table_name}').count()
adls_count = spark.sql(f'select count(*) from {adls_schema}.{table_name_adls}').count()



column mismatch, the difference {'incoming_office_region_l2', 'primary_layer', 'incept_to_issue_days', 'insured_client', 'gismo_line_of_business', 'pfo_date', 'layering_flag', 'policy_inception_date', 'underwriting_assistant', 'last12months3matured', 'quote_number', 'parent_client_name', 'modifiedhandoverdate', 'invoice_issue_date', 'underwriting_year', 'endorsement_date', 'incoming_office_region_l1', 'program_currency', 'brokeraddref', 'incoming_producing', 'run_date', 'master_broker_group', 'network_partner_instruction_date', 'true_gp', 'net_ceded_premium', 'policy_id', 'layered_policies', 'endorsement_number', 'incept_to_instruct_days', 'instruct_to_issue_days', 'fir_to_instruct_days', 'local_currency', 'policy_current_status', 'policy_identification', 'date_premium_received_by_np', 'rsr_date', 'incoming_office_type_descr', 'gismo_local_plcy_id', 'pending_due_yn', 'local_broker_group', 'global_program_number', 'edit_date', 'fir_to_issue_days', 'contract_type_code', 'pfx_date', 'prod

In [0]:
%sql
describe dw_btl_gismo.t_ax_supp_policy_details;

col_name,data_type,comment
policy_number,string,null
endorsement_number,string,null
policy_identification,string,null
policy_current_status,string,null
global_program_number,string,null
account_name,string,null
insured_client,string,null
local_broker_group,varchar(100),null
master_broker_group,varchar(100),null
captive_flag,boolean,null


In [0]:
synapse_query = "SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = 't_ax_supp_policy_gismo_local_policy' AND TABLE_SCHEMA = 'gismo_btl'"
df = run_synapse_query(synapse_query)
display(df)

COLUMN_NAME,DATA_TYPE,CHARACTER_MAXIMUM_LENGTH


In [0]:

syn_count = run_synapse_query(f'select * from {synapse_schema}.{table_name}').count()
adls_count = spark.sql(f'select * from {adls_schema}.{table_name_adls}').count()
print(syn_count)
print(adls_count)


187720
187720


In [0]:
diff_df = spark.sql(f"""
                    select policy_rate_of_exchange from gismo_btl.t_ax_supp_policy_details
                    minus 
                    select policy_rate_of_exchange from dw_btl_gismo.t_ax_supp_policy_details
                    """
)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-453372701124406>, line 1
----> 1 diff_df = spark.sql(f"""
      2                     select policy_rate_of_exchange from gismo_btl.t_ax_supp_policy_details
      3                     minus 
      4                     select policy_rate_of_exchange from dw_btl_gismo.t_ax_supp_policy_details
      5                     """
      6 )

File /databricks/spark/python/pyspark/instrumentation_utils.py:48, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     46 start = time.perf_counter()
     47 try:
---> 48     res = func(*args, **kwargs)
     49     logger.log_success(
     50         module_name, class_name, function_name, time.perf_counter() - start, signature
     51     )
     52     return res

File /databricks/spark/python/pyspark/sql/session.py:1602, in SparkSession.sql(self, sqlQuery, args, **kwargs)
   1598      

In [0]:
%sql
select * from {table_name}_vw_rs;

com.databricks.backend.common.rpc.SparkDriverExceptions$SQLExecutionException: org.apache.spark.sql.catalyst.parser.ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near '{'.(line 1, pos 14)

== SQL ==
select * from {table_name}_vw_rs
--------------^^^

	at org.apache.spark.sql.catalyst.parser.ParseException.withCommand(parsers.scala:259)
	at org.apache.spark.sql.catalyst.parser.AbstractParser.parse(parsers.scala:98)
	at org.apache.spark.sql.execution.SparkSqlParser.parse(SparkSqlParser.scala:111)
	at org.apache.spark.sql.catalyst.parser.AbstractSqlParser.parsePlan(AbstractSqlParser.scala:87)
	at com.databricks.sql.parser.DatabricksSqlParser.$anonfun$parsePlan$1(DatabricksSqlParser.scala:77)
	at com.databricks.sql.parser.DatabricksSqlParser.parse(DatabricksSqlParser.scala:98)
	at com.databricks.sql.parser.DatabricksSqlParser.parsePlan(DatabricksSqlParser.scala:74)
	at org.apache.spark.sql.SparkSession.$anonfun$sql$6(SparkSession.scala:919)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:94)
	at org.apache.spark.sql.catalyst.QueryPlanningTracker.measurePhase(QueryPlanningTracker.scala:383)
	at org.apache.spark.sql.SparkSession.$anonfun$sql$5(SparkSession.scala:918)
	at org.apache.spark.sql.SparkSession.withActive(SparkSession.scala:1148)
	at org.apache.spark.sql.SparkSession.sql(SparkSession.scala:917)
	at org.apache.spark.sql.SparkSession.$anonfun$sql$9(SparkSession.scala:951)
	at org.apache.spark.sql.SparkSession.withActive(SparkSession.scala:1148)
	at org.apache.spark.sql.SparkSession.sql(SparkSession.scala:951)
	at org.apache.spark.sql.SparkSession.sql(SparkSession.scala:984)
	at org.apache.spark.sql.SQLContext.sql(SQLContext.scala:695)
	at com.databricks.backend.daemon.driver.DriverLocal$DbClassicStrategy.executeSQLQuery(DriverLocal.scala:237)
	at com.databricks.backend.daemon.driver.DriverLocal.executeSQLSubCommand(DriverLocal.scala:319)
	at com.databricks.backend.daemon.driver.DriverLocal.$anonfun$executeSql$1(DriverLocal.scala:338)
	at scala.collection.immutable.List.map(List.scala:293)
	at com.databricks.backend.daemon.driver.DriverLocal.executeSql(DriverLocal.scala:333)
	at com.databricks.backend.daemon.driver.JupyterDriverLocal.repl(JupyterDriverLocal.scala:714)
	at com.databricks.backend.daemon.driver.DriverLocal.$anonfun$execute$34(DriverLocal.scala:1012)
	at com.databricks.unity.EmptyHandle$.runWith(UCSHandle.scala:128)
	at com.databricks.backend.daemon.driver.DriverLocal.$anonfun$execute$23(DriverLocal.scala:1003)
	at com.databricks.logging.UsageLogging.$anonfun$withAttributionContext$1(UsageLogging.scala:426)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(AttributionContext.scala:196)
	at com.databricks.logging.UsageLogging.withAttributionContext(UsageLogging.scala:424)
	at com.databricks.logging.UsageLogging.withAttributionContext$(UsageLogging.scala:418)
	at com.databricks.backend.daemon.driver.DriverLocal.withAttributionContext(DriverLocal.scala:69)
	at com.databricks.logging.UsageLogging.withAttributionTags(UsageLogging.scala:470)
	at com.databricks.logging.UsageLogging.withAttributionTags$(UsageLogging.scala:455)
	at com.databricks.backend.daemon.driver.DriverLocal.withAttributionTags(DriverLocal.scala:69)
	at com.databricks.backend.daemon.driver.DriverLocal.execute(DriverLocal.scala:958)
	at com.databricks.backend.daemon.driver.DriverWrapper.$anonfun$tryExecutingCommand$2(DriverWrapper.scala:859)
	at scala.util.Try$.apply(Try.scala:213)
	at com.databricks.backend.daemon.driver.DriverWrapper.$anonfun$tryExecutingCommand$1(DriverWrapper.scala:851)
	at com.databricks.backend.daemon.driver.DriverWrapper.$anonfun$tryExecutingCommand$3(DriverWrapper.scala:891)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:667)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:685)
	at com.databricks.logging.UsageLogging.$anon

In [0]:
df = spark.sql("Select net_ceded_premium from dw_btl_gismo.t_ax_supp_policy_details");
display(df)

net_ceded_premium
2975.000000000000000000
14337.990234375000000000
2700.000000000000000000
2125.000000000000000000
116070.046875000000000000
-647.190002441406200000
7943.609863281250000000
645537.125000000000000000
1700.000000000000000000
1703.750000000000000000


In [0]:
df = spark.sql(f"SELECT net_ceded_premium FROM {table_name}_vw_rs");
display(df)

net_ceded_premium
2625.000000000000000000
2700.000000000000000000
1275.000000000000000000
2975.000000000000000000
4250.000000000000000000
2125.000000000000000000
6882.390136718750000000
2550.000000000000000000
5451.589843750000000000
2454.800048828125000000


In [0]:
display(diff_df)

Claim Identifier,Claim Number,Global Claim Number,Claim Incident Date,Claim Cause of Loss Description,Claim Status Description,Planning Line of Business Name,Large Catastrophe Attritional Threat Code,Claim Event Code,Claim Event Name,Bulk Bordereaux Indicator
22821886,M990902747,0000125317,1900-01-01,Unknown,,Unknown,CAT,UNM,Unmapped,
23024578,N40020680300,,1900-01-01,Unknown,,Unknown,CAT,UKN,Unknown,
22698539,N60053630103,0000197992,1900-01-01,Unknown,,Unknown,CAT,UNM,Unmapped,
22715753,N60051550302,0000196631,1900-01-01,Unknown,,Unknown,Large,UNM,Unmapped,
23324995,N60163691101,0000244569,1900-01-01,Unknown,,Unknown,Large,UNM,Unmapped,
23134235,A61026640998,0000032199,1900-01-01,Unknown,,Unknown,Large,UNM,Unmapped,
23422389,A61018540901,0000029632,1900-01-01,Unknown,,Unknown,Large,UNM,Unmapped,
23132401,A60008560204,0000012910,1900-01-01,Unknown,,Unknown,Large,UNM,Unmapped,
22940143,N60062651199,0000203630,1900-01-01,Unknown,,Unknown,CAT,UNM,Unmapped,
22933820,M020103082,,1900-01-01,Unknown,,Unknown,Large,UKN,Unknown,


In [0]:
%sql
select * from btl_src.art_claims limit 10;

Claim_currency,Claim_handling_office,Claim_title,ClaimantKey,ClaimHandlerKey,ClaimKey,ClaimReference,ClaimsCauseCategoryKey,ClaimStatusKey,Date_of_closing,Date_of_loss,Date_of_notification,dAxaShareOfTotal,dBase2ClaimExchangeRate,dCurrentExpenses,dCurrentExpensesLegal,dCurrentExpensesReserve,dCurrentIndemnity,dCurrentIndemnityReserve,dCurrentPaidTotal,dCurrentReserveTotal,dGrossTotalIncurred,dInitialExpensesReserve,dInitialIndemnityReserve,dInitialReserve,dRecovery,dtpolicyperiodfrom,EventKey,Last_payment,Loss_location_country,LossTypeKey,OfficeKey,PolicyKey,PolicyLocationKey,PolicyReference,sEventCode
EUR,Paris,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX-X,31311,45658,11929,0080212,11,2054,2001-11-19T00:00:00Z,2000-09-19T00:00:00Z,2000-09-26T00:00:00Z,100.000000,1.00000000,364.660000,364.660000,0.000000,0.000000,0.000000,364.660000,0.000000,364.660000,0.00,14483.00,14483.00,null,2004-01-01T00:00:00Z,null,2001-11-19T00:00:00Z,null,null,7,58544,null,808008,null
EUR,Paris,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX-X,33671,45658,12256,0080213,11,2118,2002-10-04T00:00:00Z,2000-10-03T00:00:00Z,2000-10-06T00:00:00Z,100.000000,1.00000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,152.00,152.00,null,2000-03-21T00:00:00Z,null,2002-10-04T00:00:00Z,null,null,7,65268,null,840944,null
EUR,Paris,Date de cr?ation du dossier : 06/11/2000,33879,45658,11685,0080214,11,2054,2000-11-22T00:00:00Z,2000-10-23T00:00:00Z,2000-11-03T00:00:00Z,100.000000,1.00000000,364.660000,364.660000,0.000000,0.000000,0.000000,364.660000,0.000000,364.660000,0.00,6098.00,6098.00,null,2005-08-01T00:00:00Z,null,2000-11-22T00:00:00Z,null,null,7,60869,null,811162,null
EUR,Paris,Date de cr?ation du dossier : 13/11/2000,31483,45658,11446,0080215,11,2054,2000-12-04T00:00:00Z,2000-10-13T00:00:00Z,2000-10-18T00:00:00Z,100.000000,1.00000000,0.000000,0.000000,0.000000,274.410000,0.000000,274.410000,0.000000,274.410000,0.00,274.00,274.00,null,2004-12-01T00:00:00Z,null,2000-12-04T00:00:00Z,null,null,7,58687,null,808267,null
EUR,Paris,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX-X,31877,45658,11447,0080216,11,2054,2006-10-16T00:00:00Z,2000-10-03T00:00:00Z,2000-10-18T00:00:00Z,100.000000,1.00000000,3152.800000,2554.800000,0.000000,18000.000000,0.000000,21152.800000,0.000000,21152.800000,0.00,7622.00,7622.00,null,2000-12-27T00:00:00Z,null,2006-10-16T00:00:00Z,null,null,7,59144,null,809086,null
EUR,Paris,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX-X,32782,45658,11817,0080217,11,2118,2003-10-16T00:00:00Z,2000-03-14T00:00:00Z,2000-10-09T00:00:00Z,100.000000,1.00000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,7622.00,7622.00,null,1999-05-01T00:00:00Z,null,2003-10-16T00:00:00Z,null,null,7,59848,null,810037,null
EUR,Paris,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX-X,33092,45658,12001,0080218,11,2054,2001-01-10T00:00:00Z,2000-10-23T00:00:00Z,2000-10-30T00:00:00Z,100.000000,1.00000000,0.000000,0.000000,0.000000,503.080000,0.000000,503.080000,0.000000,503.080000,0.00,1067.00,1067.00,null,2002-04-01T00:00:00Z,null,2001-01-10T00:00:00Z,null,null,7,60288,null,810533,null
EUR,Paris,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX-X,33239,45658,10124,0080219,11,2118,2000-12-12T00:00:00Z,2000-09-01T00:00:00Z,2000-10-30T00:00:00Z,100.000000,1.00000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,3811.00,3811.00,null,2000-06-01T00:00:00Z,null,2000-12-12T00:00:00Z,null,null,7,60373,null,810626,null
EUR,Paris,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX-X,33321,45658,11819,0080220,11,2118,2004-04-02T00:00:00Z,2000-10-24T00:00:00Z,2000-11-02T00:00:00Z,100.000000,1.00000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,152.00,152.00,null,2002-06-01T00:00:00Z,null,2004-04-02T00:00:00Z,null,null,7,60783,null,811070,null
EUR,Paris,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX-X,30818,45658,12254,0080221,11,2118,2003-08-12T00:00:00Z,2000-10-15T00:00:00Z,2000-10-26T00:00:00Z,100.000000,1.00000000,0.000000,0.000000,0.000000,0